# HW1 — полный запуск в Google Colab

Этот ноутбук — только интерфейс запуска. Он **не дублирует реализацию**: тесты, измерения, калибровка и графики выполняются существующими файлами `tests/test_hw1.py`, `measure.py`, `calibrate.py` и `plots.py`. Поэтому проект по-прежнему можно запускать из терминала.

Перед началом выберите в Colab: **Runtime → Change runtime type → T4 GPU**.

Запускайте ячейки строго сверху вниз. Полный `measure.py` умеет продолжать прерванный запуск: если среда Colab отключилась, подключите T4 снова и повторно выполните ноутбук. При использовании Google Drive уже записанные строки CSV сохранятся.

## 1. Получить файлы проекта

Рекомендуемый режим сохраняет проект в Google Drive, чтобы не потерять промежуточные измерения при отключении Colab.

Есть два способа: 

1. После публикации репозитория вставить его адрес в `REPO_URL`.
2. Оставить `REPO_URL` пустым и загрузить ZIP-архив папки HW1, когда появится окно выбора файла.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import zipfile

USE_GOOGLE_DRIVE = True
REPO_URL = ""  # После публикации: https://github.com/<user>/<repo>.git
BRANCH = "main"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    base_dir = Path("/content/drive/MyDrive/ITMO")
else:
    base_dir = Path("/content")

base_dir.mkdir(parents=True, exist_ok=True)
expected_dir = base_dir / "HW1"

def is_project(path: Path) -> bool:
    required = ("models.py", "equations.py", "measure.py", "calibrate.py", "plots.py")
    return path.is_dir() and all((path / name).is_file() for name in required)

if is_project(expected_dir):
    PROJECT_DIR = expected_dir
    print(f"Используем уже существующий проект: {PROJECT_DIR}")
elif REPO_URL.strip():
    if expected_dir.exists():
        raise RuntimeError(f"Папка {expected_dir} уже существует, но не похожа на HW1. Переименуйте или удалите её вручную.")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL.strip(), str(expected_dir)],
        check=True,
    )
    PROJECT_DIR = expected_dir
else:
    from google.colab import files
    print("Выберите ZIP-архив папки HW1")
    uploaded = files.upload()
    zip_items = [(name, data) for name, data in uploaded.items() if name.lower().endswith(".zip")]
    if len(zip_items) != 1:
        raise RuntimeError("Нужно загрузить ровно один ZIP-архив проекта HW1.")
    archive_name, archive_data = zip_items[0]
    archive_path = Path("/content") / archive_name
    archive_path.write_bytes(archive_data)
    unpack_dir = Path("/content/hw1_uploaded")
    if unpack_dir.exists():
        shutil.rmtree(unpack_dir)
    unpack_dir.mkdir(parents=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(unpack_dir)
    candidates = sorted({path.parent for path in unpack_dir.rglob("models.py") if is_project(path.parent)})
    if len(candidates) != 1:
        raise RuntimeError(f"В архиве ожидалась одна папка HW1, найдено: {candidates}")
    shutil.copytree(candidates[0], expected_dir, dirs_exist_ok=True)
    PROJECT_DIR = expected_dir

assert is_project(PROJECT_DIR)
print(f"PROJECT_DIR={PROJECT_DIR}")

## 2. Установить зависимости

Colab уже содержит PyTorch с CUDA. Команда ниже устанавливает только отсутствующие или несовместимые зависимости из `requirements.txt`.

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)
print("Зависимости установлены")

## 3. Проверить GPU и окружение

Ячейка остановит выполнение, если GPU недоступна или имеет больше 16 ГБ. Для итоговой работы нужна именно T4/P100 или другая NVIDIA GPU не более 16 ГБ.

In [ ]:
subprocess.run(["nvidia-smi"], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA недоступна. Включите T4: Runtime → Change runtime type → T4 GPU.")

properties = torch.cuda.get_device_properties(0)
total_gib = properties.total_memory / 2**30
print(f"GPU: {properties.name}")
print(f"GPU memory: {total_gib:.2f} GiB")
print(f"PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}; cuDNN: {torch.backends.cudnn.version()}")

if total_gib > 16.5:
    raise RuntimeError("Эта GPU больше 16 ГБ и не соответствует условию. Выберите T4/P100.")

## 4. Запустить тесты

Ноутбук вызывает обычный Python-файл тестов. Ожидается пять тестов со статусом `OK`.

In [ ]:
def run_project(*arguments: str) -> None:
    command = [sys.executable, "-u", *arguments]
    print("$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)

run_project("-m", "unittest", "discover", "-s", "tests", "-v")

## 5. Короткая smoke-проверка

Она запускает `measure.py` только на трёх маленьких точках, включая energy. Результат сохраняется как `results/measurements_smoke.csv` и не используется в финальном анализе.

In [ ]:
RUN_SMOKE = True

if RUN_SMOKE:
    run_project("measure.py", "--smoke", "--overwrite")
else:
    print("Smoke-проверка пропущена")

## 6. Полные 132 измерения

Это главный и самый долгий этап. Выполняется существующий `measure.py`. Каждая завершённая строка сразу добавляется в CSV. При повторном запуске уже готовые `(S,B)` пропускаются.

Оставьте `OVERWRITE_FULL_RUN=False`, чтобы продолжать прерванный запуск. Установите `True` только если осознанно хотите удалить старый `measurements.csv` и начать заново.

In [ ]:
OVERWRITE_FULL_RUN = False

arguments = ["measure.py"]
if OVERWRITE_FULL_RUN:
    arguments.append("--overwrite")
run_project(*arguments)

## 7. Проверить промежуточный CSV

Здесь отображаются первые строки, количество `ok/OOM` и разбиение calibration/validation. Если строк меньше 132, повторно запустите предыдущую ячейку.

In [ ]:
import pandas as pd
from IPython.display import display

measurements_path = PROJECT_DIR / "results" / "measurements.csv"
measurements = pd.read_csv(measurements_path)
display(measurements.head(10))
print("Строк данных:", len(measurements))
print("Уникальных (S,B):", measurements[["image_size", "batch"]].drop_duplicates().shape[0])
print("Статусы:")
display(measurements["status"].value_counts(dropna=False).rename_axis("status").to_frame("count"))
print("Calibration/validation:")
display(measurements["is_validation"].value_counts(dropna=False).rename_axis("is_validation").to_frame("count"))

if len(measurements) != 132:
    raise RuntimeError(f"Получено {len(measurements)} из 132 строк. Повторно запустите полное измерение.")
if measurements[["image_size", "batch"]].duplicated().any():
    raise RuntimeError("В CSV найдены повторяющиеся пары (S,B).")
print("CSV содержит все 132 уникальные конфигурации")

## 8. Калибровать latency и energy

Выполняется существующий `calibrate.py`. Он использует calibration-точки, сохраняет параметры в `theta.json` и считает метрики отдельно для calibration и validation.

In [ ]:
run_project("calibrate.py")

In [ ]:
import json
from pprint import pprint

theta = json.loads((PROJECT_DIR / "results" / "theta.json").read_text())
summary = json.loads((PROJECT_DIR / "results" / "analysis_summary.json").read_text())
print("Подобранные параметры:")
pprint(theta)
print("\nМетрики:")
pprint(summary)

## 9. Построить и показать графики

Выполняется существующий `plots.py`, после чего PNG отображаются прямо в ноутбуке.

In [ ]:
run_project("plots.py")

In [ ]:
from IPython.display import Image, Markdown, display

figure_dir = PROJECT_DIR / "results" / "figures"
figure_paths = sorted(figure_dir.glob("*.png"))
if len(figure_paths) != 5:
    raise RuntimeError(f"Ожидалось 5 PNG, найдено {len(figure_paths)}")
for figure_path in figure_paths:
    display(Markdown(f"### `{figure_path.name}`"))
    display(Image(filename=str(figure_path)))

## 10. Скачать результаты

Ячейка упакует папку `results/` в ZIP и предложит скачать её. Если проект находится в Google Drive, результаты уже сохранены там, но отдельный архив всё равно полезен как резервная копия.

In [ ]:
archive_base = Path("/content/hw1_results")
archive_path = Path(shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=PROJECT_DIR,
    base_dir="results",
))
print(f"Создан архив: {archive_path} ({archive_path.stat().st_size / 2**20:.2f} MiB)")

from google.colab import files
files.download(str(archive_path))

## Что остаётся после ноутбука

Ноутбук получает экспериментальные результаты, параметры и графики. После него всё ещё нужно:

1. перенести `results/` в основной локальный репозиторий, если запуск был в отдельной копии;
2. заполнить README фактическими числами и обсуждением;
3. написать вывод формул от руки и добавить `hw1_handwritten.pdf`;
4. проверить репозиторий и отправить ссылку преподавателю.